In [0]:
%sql
INSERT OVERWRITE TABLE business_intelligence.brandcollab_e2e (
with creator_base as (
select distinct creator_id, id as varient_creator_id, varient_id,campaign_status, created_at creator_initiated_at
from oldmonkey_production.bronze_brandcollab_varientcreatormapping as vcm
),

--address and quotedprice cte
brandcollab_creatorfieldmapping as (
    select * from oldmonkey_production.bronze_brandcollab_creatorfieldmapping
),

status as (
    select distinct id as varient_creator_id, status, rejection_reason, delivery_status, date(updated_at) as period
    from oldmonkey_production.bronze_brandcollab_varientcreatormapping
),

pitched_price as (
    select varient_creator_id, pitched_price, period, pitched_price_done_by
    from (
    select varient_creator_id,
           varient_field_value as pitched_price,
           date(cfm.updated_at) as period, 
           cfm.updated_by pitched_price_done_by,
           row_number() over(partition by varient_creator_id order by DATE(cfm.updated_at) desc) rn
    from oldmonkey_production.bronze_brandcollab_creatorfieldmapping cfm
    join oldmonkey_production.bronze_brandcollab_varientfieldmapping as vfm
      on cfm.varient_field_id=vfm.id
    where vfm.field_id=15 
    )
    where rn = 1
),

acknowledged as (
    select varient_creator_id, acknowledge, ack_period, ack_done_by
    from (
    select varient_creator_id,
          varient_field_value as acknowledge,
          date(cfm.updated_at) as ack_period,
          cfm.updated_by ack_done_by,
          row_number() over(partition by varient_creator_id order by DATE(cfm.updated_at) desc) rn
    from oldmonkey_production.bronze_brandcollab_creatorfieldmapping cfm
    join oldmonkey_production.bronze_brandcollab_varientfieldmapping as vfm
      on cfm.varient_field_id=vfm.id
    where vfm.field_id=1
    )
    where rn = 1
),

opt as (
    select varient_creator_id, opted_in, optin_period, opted_by
    from (
    select varient_creator_id,varient_field_value as opted_in,date(cfm.updated_at) as optin_period,
    cfm.updated_by opted_by,
    row_number() over(partition by varient_creator_id order by date(cfm.updated_at) desc) rn
    from oldmonkey_production.bronze_brandcollab_creatorfieldmapping cfm
    join oldmonkey_production.bronze_brandcollab_varientfieldmapping as vfm
      on cfm.varient_field_id=vfm.id
    where vfm.field_id=13 and varient_field_value like '%Yes%'
    )
    where rn = 1
),
final_price as (
    select varient_creator_id, final_price, final_price_date, final_price_done_by
    from (
    select varient_creator_id,
           CASE 
            WHEN varient_field_value RLIKE '^[-+]?[0-9]*\\.?[0-9]+$' 
            THEN CAST(varient_field_value AS FLOAT) 
            ELSE NULL 
        END AS final_price,
           date(cfm.updated_at) as final_price_date,
        cfm.updated_by final_price_done_by,
        row_number() over(partition by varient_creator_id order by date(cfm.updated_at) desc) rn
    from oldmonkey_production.bronze_brandcollab_creatorfieldmapping cfm
    join oldmonkey_production.bronze_brandcollab_varientfieldmapping as vfm
      on cfm.varient_field_id=vfm.id and varient_field_value is not null
    where vfm.field_id=14
    )
    where rn = 1
),
creator_selection as (
    select varient_creator_id,varient_field_value as creator_selected,date(cfm.updated_at) as period,
    cfm.updated_by creator_selected_by
    from oldmonkey_production.bronze_brandcollab_creatorfieldmapping cfm
    join oldmonkey_production.bronze_brandcollab_varientfieldmapping as vfm
      on cfm.varient_field_id=vfm.id
    where vfm.field_id=12 and varient_field_value like '%Yes%'
),
script_allowed as (
    SELECT vfm.varient_id, varient_creator_id, varient_field_value as script_allowed,
    cfm.updated_by script_submitted_by
    FROM oldmonkey_production.bronze_brandcollab_creatorfieldmapping cfm
    JOIN oldmonkey_production.bronze_brandcollab_varientfieldmapping AS vfm
      ON cfm.varient_field_id = vfm.id
    WHERE vfm.field_id = 157
),
script_submission as (
    select variant_creator_id, min(date(created_at)) script_submission_date
    from oldmonkey_production.bronze_brandcollab_variantscriptsubmissions
    group by all
),
script_status as (
    select variant_creator_id, status script_status, updated_at script_latest_update_date
    from oldmonkey_production.bronze_brandcollab_variantscriptsubmissions
    where is_latest = true
),
-- script_approved as (
-- select distinct variant_creator_id, min(date(updated_at)) script_approved_date
-- from oldmonkey_production.bronze_brandcollab_variantscriptsubmissions
-- where status = 'APPROVED' 
-- group by all
-- ),
final_script as (
    select a.*, b.script_submission_date, script_status
    -- , sa.script_approved_date
    from script_allowed a
    left join script_submission b
    on a.varient_creator_id=b.variant_creator_id
    left join script_status c
    on a.varient_creator_id=c.variant_creator_id
    -- left join script_approved sa 
    -- on a.varient_creator_id=sa.variant_creator_id
)
,
product_delivery as (
    select varient_creator_id,varient_field_value as product_delivery_status,date(cfm.updated_at) as period
    from oldmonkey_production.bronze_brandcollab_creatorfieldmapping cfm
    join oldmonkey_production.bronze_brandcollab_varientfieldmapping as vfm
      on cfm.varient_field_id=vfm.id
    where vfm.field_id=4 
),

product_receiving as (
    select varient_creator_id,varient_field_value as product_receiving,date(cfm.updated_at) as period
    from oldmonkey_production.bronze_brandcollab_creatorfieldmapping cfm
    join oldmonkey_production.bronze_brandcollab_varientfieldmapping as vfm
      on cfm.varient_field_id=vfm.id
    where vfm.field_id=3 and varient_field_value='true'
),

draft_status1 as (
    select 
        varient_creator_id,
        COUNT(DISTINCT CASE WHEN cvd.status IN ('DRAFT_PENDING', 'DRAFT_APPROVED', 'DRAFT_REJECTED') THEN cvd.id END) AS Draft_submitted,
        COUNT(DISTINCT CASE WHEN cvd.status = 'DRAFT_APPROVED' THEN cvd.id END) AS Draft_approved,
        COUNT(DISTINCT CASE WHEN cvd.status = 'DRAFT_REJECTED' THEN cvd.id END) AS Draft_rejected
    from oldmonkey_production.bronze_brandcollab_creatorvarientdeliverable cvd
    group by 1
),

draft_status2 as (
    select varient_creator_id, 
           max(case when status like '%APPROVED%' then updated_at end) as latest_approved, 
           max(case when status like '%PENDING%' then updated_at end) as latest_pending,
           max(case when status like '%REJECTED%' then updated_at end) as latest_rejected
    from oldmonkey_production.bronze_brandcollab_creatorvarientdeliverable cvd
    group by 1
),

draft_status3 as (
    select 
        cvd.varient_creator_id, 
        cvd.rejection_reason as rejection_reason
    from oldmonkey_production.bronze_brandcollab_creatorvarientdeliverable cvd
    join draft_status2 as ds
      on ds.varient_creator_id=cvd.varient_creator_id and ds.latest_rejected=cvd.updated_at
),
content_live2 as (
    select varient_creator_id,
           max(case when status like '%APPROVED%' then concat(updated_at,'-',url) end) as latest_approved, 
           max(case when status like '%PENDING%' then concat(updated_at,'-',url) end) as latest_pending,
           max(case when status like '%REJECTED%' then concat(updated_at,'-',url) end) as latest_rejected
    from oldmonkey_production.bronze_brandcollab_creatorcontentsubmission cvd
    group by all
),
content_live_done_by as (
select varient_creator_id, content_submitted_by
from
(
    select varient_creator_id, added_by content_submitted_by, 
    row_number() over(partition by varient_creator_id order by DATE(updated_at) desc, id desc) rn
    from oldmonkey_production.bronze_brandcollab_creatorcontentsubmission cvd
    where status ilike '%APPROVED%' or status ilike '%PENDING%' or status like '%REJECTED%'
)
where rn =1
),
remark as (
    select varient_creator_id,
           varient_field_value as remark,
           date(cfm.updated_at) as period,
           cfm.updated_by remark_added_by
    from oldmonkey_production.bronze_brandcollab_creatorfieldmapping cfm
    join oldmonkey_production.bronze_brandcollab_varientfieldmapping as vfm
      on cfm.varient_field_id=vfm.id
    where vfm.field_id=18
),

invoice_link as (
    SELECT vfm.varient_id, varient_creator_id, varient_field_value as invoice_link,
    cfm.updated_by invoice_submitted_by
    FROM oldmonkey_production.bronze_brandcollab_creatorfieldmapping cfm
    JOIN oldmonkey_production.bronze_brandcollab_varientfieldmapping AS vfm
      ON cfm.varient_field_id = vfm.id
    WHERE vfm.field_id =23
),

invoice_status as (
    SELECT vfm.varient_id, varient_creator_id, varient_field_value as invoice_status
    FROM oldmonkey_production.bronze_brandcollab_creatorfieldmapping cfm
    JOIN oldmonkey_production.bronze_brandcollab_varientfieldmapping AS vfm
      ON cfm.varient_field_id = vfm.id
    WHERE vfm.field_id =24
),

draft_status_summary AS (
    SELECT 
        varient_creator_id,
        -- changed due to status keeps changing and only pending draft will be shown as submitted

        -- SUM(CASE WHEN status LIKE '%PENDING%' THEN 1 ELSE 0 END) AS draft_submitted,
        -- MAX(CASE WHEN status LIKE '%PENDING%' THEN DATE(updated_at) END) AS draft_submitted_date,
        SUM(CASE WHEN status in ('DRAFT_APPROVED', 'DRAFT_PENDING', 'DRAFT_REJECTED') THEN 1 ELSE 0 END) AS draft_submitted,
        MAX(CASE WHEN status in ('DRAFT_APPROVED', 'DRAFT_PENDING', 'DRAFT_REJECTED') THEN DATE(created_at) END) AS draft_submitted_date,
        SUM(CASE WHEN status LIKE '%APPROVED%' THEN 1 ELSE 0 END) AS draft_approved,
        MAX(CASE WHEN status LIKE '%APPROVED%' THEN DATE(updated_at) END) AS draft_approved_date
    FROM oldmonkey_production.bronze_brandcollab_creatorvarientdeliverable
    GROUP BY all
),
draft_added_by as (
select varient_creator_id, added_by draft_added_by  
from (
select *, 
row_number() over(partition by varient_creator_id order by DATE(updated_at) desc, id desc) rn
FROM oldmonkey_production.bronze_brandcollab_creatorvarientdeliverable
)
where rn = 1
),
content_approved AS (
    SELECT 
        varient_creator_id,
        
        -- changed due to status keeps changing and only pending draft will be shown as submitted
        -- SUM(DISTINCT CASE WHEN status LIKE '%PENDING%' THEN 1 ELSE 0 END) AS content_submitted,
        -- MAX(DISTINCT CASE WHEN status LIKE '%PENDING%' THEN DATE(updated_at) END) content_submitted_date,
        SUM(DISTINCT CASE WHEN status in ('CONTENT_PENDING', 'CONTENT_REJECTED', 'CONTENT_APPROVED') THEN 1 ELSE 0 END) AS content_submitted,
        MAX(DISTINCT CASE WHEN status in ('CONTENT_PENDING', 'CONTENT_REJECTED', 'CONTENT_APPROVED') THEN DATE(created_at) END) AS content_submitted_date,
        SUM(DISTINCT CASE WHEN status LIKE '%APPROVED%' THEN 1 ELSE 0 END) AS content_approved,
        MAX(DISTINCT CASE WHEN status LIKE '%APPROVED%' THEN DATE(updated_at) END) AS content_approved_date
    FROM oldmonkey_production.bronze_brandcollab_creatorcontentsubmission
    GROUP BY 1
),
content_approved_by as (
select varient_creator_id, added_by content_approved_by  
from (
select *, 
row_number() over(partition by varient_creator_id order by DATE(updated_at) desc, id desc) rn
FROM oldmonkey_production.bronze_brandcollab_creatorcontentsubmission
)
where rn = 1
),
payout as (
    select 
        varient_creator_id,
        count(varient_creator_id) as payout_count,
        sum(payout_amount) as payouts_given,
        ARRAY_JOIN(collect_list(concat(date(payout_date)," ",round(payout_amount,0)," ",payout_medium," ",payout_utr," ")),',') payout_meta
    from oldmonkey_production.bronze_brandcollab_creatorcampignpayout
    where is_alive=true
    group by all
),

-- renamed cte1 → cte_payouts
cte_payouts AS (
    SELECT 
        cv.campaign_id,
        c.name AS campaign_name,
        vcm.varient_id,
        cv.name AS varient_name,
        vcm.creator_id,
        ac.username,
        cb.varient_creator_id,
        is.invoice_status,
        il.invoice_link,
        COALESCE(fp.final_price, 0) AS final_price,
        -- ss.script_allowed,
        -- ss.script_submission_date,
        -- ss.script_submitted_by,
        -- ss.script_status,
        COALESCE(ds.draft_submitted, 0) AS draft_submitted,
        COALESCE(ds.draft_approved, 0) AS draft_approved,
        ds.draft_submitted_date AS draft_submitted_date,
        ds.draft_approved_date AS draft_approved_date,
        dab.draft_added_by,
        COALESCE(ca.content_submitted, 0) AS content_submitted,
        COALESCE(ca.content_approved, 0) AS content_approved,
        content_submitted_date,
        content_approved_date,
        content_approved_by,
        p.payouts_given
        ---action done by
        -- pitched_price_done_by,
        -- ack_done_by,
        -- opted_by,
        -- final_price_done_by,
        -- creator_selected_by,
        -- script_submitted_by,
        -- draft_added_by,
        -- remark_added_by,
        -- invoice_submitted_by
    FROM creator_base cb
    LEFT JOIN final_price fp ON cb.varient_creator_id = fp.varient_creator_id
    -- LEFT JOIN final_script ss ON cb.varient_creator_id = ss.varient_id
    LEFT JOIN draft_status_summary ds ON cb.varient_creator_id = ds.varient_creator_id
    LEFT JOIN draft_added_by dab ON cb.varient_creator_id = dab.varient_creator_id
    LEFT JOIN content_approved_by cab ON cb.varient_creator_id = cab.varient_creator_id
    LEFT JOIN content_approved ca ON cb.varient_creator_id = ca.varient_creator_id
    LEFT JOIN payout p ON cb.varient_creator_id = p.varient_creator_id
    left join invoice_link il on cb.varient_creator_id = il.varient_creator_id
    left join invoice_status is on cb.varient_creator_id = is.varient_creator_id
    LEFT JOIN oldmonkey_production.bronze_brandcollab_varientcreatormapping vcm ON cb.varient_creator_id = vcm.id
    LEFT JOIN oldmonkey_production.bronze_brandcollab_campaignvarient cv ON vcm.varient_id = cv.id
    LEFT JOIN oldmonkey_production.bronze_brandcollab_campaign c ON c.id = cv.campaign_id
    LEFT JOIN oldmonkey_production.bronze_atg_creator ac ON ac.id = vcm.creator_id
    WHERE fp.final_price > 0
),

-- renamed base → base_main
base_main AS (
    SELECT 
        campaign_id,
        campaign_name,
        varient_id,
        varient_name,
        creator_id,
        username,
        varient_creator_id,
        invoice_status,
        invoice_link,
        final_price,
        -- script_allowed,
        -- script_submitted_by,
        -- script_submission_date,
        -- script_status,
        draft_approved,
        draft_submitted,
        draft_added_by,
        draft_submitted_date,
        draft_approved_date,
        content_approved,
        content_submitted,
        content_approved_by,
        content_submitted_date,
        content_approved_date,
        COALESCE(payouts_given, 0) AS payouts_given
        
        -- pitched_price_done_by,
        -- ack_done_by,
        -- opted_by,
        -- final_price_done_by,
        -- creator_selected_by,
        -- draft_added_by,
        -- remark_added_by,
        -- invoice_submitted_by
    FROM cte_payouts
),

-- renamed base2 → base_with_tbd
base_with_tbd AS (
    SELECT *,
        CASE 
            WHEN payouts_given = final_price THEN 0
            WHEN content_approved = 0 AND draft_approved > 0 AND payouts_given > 0 THEN 0
            WHEN content_approved > 0 AND payouts_given = 0 THEN final_price
            WHEN content_approved > 0 AND payouts_given > 0 AND payouts_given < final_price THEN final_price - payouts_given
            WHEN content_approved = 0 AND draft_approved > 0 THEN 0.3 * final_price
            ELSE 0 
        END AS payout_tbd
    FROM base_main
),
ad_rights as (
    SELECT 
        c.id AS campaign_id,
        c.name AS campaign_name,
        cv.name AS varient_name,
        vcm.varient_id,
        mcp.brandcollab_variant_creator_mapping_id AS varient_creator_id,
        mcp.creator_id,
        COALESCE(fp.final_price, 0) AS final_price,
        COALESCE(ds.draft_approved, 0) AS draft_approved,
        COALESCE(ca.content_approved, 0) AS content_approved,
        CONCAT('[', ARRAY_JOIN(COLLECT_LIST(CAST(DATE(mcp.added_on) AS STRING)), ','), ']') AS ad_date_string,
        '[' || STRING_AGG(CAST(mcp.payout_id AS VARCHAR(100)), ',') || ']' AS payout_id,
        '[' || STRING_AGG(mcp.status, ',') || ']' AS status1,
        '[' || STRING_AGG(pd.status, ',') || ']' AS status2,
        SUM(
            CASE 
                WHEN mcp.amount RLIKE '^[-+]?[0-9]*\\.?[0-9]+$' 
                THEN CAST(mcp.amount AS FLOAT) 
                ELSE 0 
            END
        ) AS amount,
        '[' || STRING_AGG(pr.transaction_utr, ',') || ']' AS misc_transaction_utr
    FROM oldmonkey_production.bronze_atg_miscellaneouscreatorpayouts mcp
    JOIN oldmonkey_production.bronze_atg_payoutdetails pd 
        ON mcp.payout_id = pd.id
    JOIN oldmonkey_production.bronze_atg_payoutrazorpay pr 
        ON pd.payout_id = pr.razorpay_payout_id
    JOIN oldmonkey_production.bronze_brandcollab_varientcreatormapping vcm 
        ON vcm.id = mcp.brandcollab_variant_creator_mapping_id
    JOIN oldmonkey_production.bronze_brandcollab_campaignvarient cv 
        ON vcm.varient_id = cv.id
    JOIN oldmonkey_production.bronze_brandcollab_campaign c 
        ON c.id = cv.campaign_id
    LEFT JOIN final_price fp 
        ON mcp.brandcollab_variant_creator_mapping_id = fp.varient_creator_id
    LEFT JOIN draft_status_summary ds 
        ON mcp.brandcollab_variant_creator_mapping_id = ds.varient_creator_id
    LEFT JOIN content_approved ca 
        ON mcp.brandcollab_variant_creator_mapping_id = ca.varient_creator_id
    WHERE entry_type = 'BRAND_COLLAB' 
      AND pd.status = 'PAID' 
      AND razorpay_payout_status = 'processed'
      and lower(mcp.name) like ('%ad right%')
    GROUP BY all
),


-- renamed base3 → base_payouts_extended
base_payouts_extended AS (
    SELECT 
        b2.campaign_id,
        b2.campaign_name,
        b2.varient_name,
        b2.varient_id,
        b2.varient_creator_id,
        b2.creator_id,
        b2.username,
        b2.invoice_status,
        b2.invoice_link,
        b2.final_price,
        -- b2.script_allowed,
        -- b2.script_submission_date,
        -- b2.script_submitted_by,
        -- b2.script_status,
        b2.draft_approved,
        b2.draft_submitted,
        b2.draft_added_by,
        b2.draft_submitted_date,
        b2.draft_approved_date,
        b2.content_approved,
        b2.content_submitted,
        b2.content_approved_by,
        content_submitted_date,
        content_approved_date,
        COALESCE(b2.payouts_given, 0) AS payouts_given,
        b2.payout_tbd AS payout_tbd,
        p1.payout_meta,
        concat(
            '{"payout_id": "', ar.payout_id, 
            '", "amount": ', ar.amount, 
            '", "misc_transaction_utr": "', ar.misc_transaction_utr,
            '", "ad_date_string": "', ar.ad_date_string,'"}'
        ) as ad_right_json
    FROM base_with_tbd b2
    LEFT JOIN payout p1 ON p1.varient_creator_id = b2.varient_creator_id
    left join ad_rights ar on ar.varient_creator_id=b2.varient_creator_id
),

poc as (
 select varient_id, default_value as poc
 from oldmonkey_production.bronze_brandcollab_varientfieldmapping as vfm
 where vfm.field_id=19
),

final_payouts_cte as (
    SELECT 
        campaign_id,
        campaign_name,
        varient_name,
        varient_id,
        varient_creator_id,
        creator_id,
        username,
        invoice_status,
        invoice_link,
        final_price,
        -- script_allowed,
        -- script_submission_date,
        -- script_submitted_by,
        -- script_status,
        draft_submitted,
        draft_approved,
        draft_added_by,
        draft_submitted_date,
        draft_approved_date,
        content_submitted,
        content_approved,
        content_approved_by,
        content_submitted_date,
        content_approved_date,
        payouts_given,
        payout_tbd,
        payout_meta,
        ad_right_json
    FROM base_payouts_extended
)
,

-- renamed cte1 → cte_content
cte_content AS (
    SELECT DISTINCT
      cv.id AS varient_id,
      cv.name AS varient_name,
      c.id as campaign_id,
      c.name as campaign_name,
      ccs.varient_creator_id AS varient_creator_id,
      ccs.status AS status,
      ccs.url AS post_url,
      DATE(CAST(ccs.updated_at AS TIMESTAMP)) AS period,
      ccs.post_id as post_id
    FROM oldmonkey_production.bronze_brandcollab_creatorcontentsubmission ccs
    INNER JOIN oldmonkey_production.bronze_brandcollab_varientcreatormapping vcm
      ON ccs.varient_creator_id = vcm.id
    INNER JOIN oldmonkey_production.bronze_brandcollab_campaignvarient cv
      ON vcm.varient_id = cv.id
    inner join oldmonkey_production.bronze_brandcollab_campaign c
      ON cv.campaign_id = c.id
    WHERE ccs.status='CONTENT_APPROVED' 
    -- and c.is_paid=true
)
,

-- downstream references now should use cte_content, base_main, base_with_tbd, base_payouts_extended
-- (continue with your post_id, sales, clicks, final_post etc. unchanged, just referencing new names)



post_id AS (
SELECT DISTINCT
  c.varient_id,
  c.varient_name,
  c.campaign_id,
  c.campaign_name,
  c.varient_creator_id,
  c.status as status,
  c.post_url,
  c.period,
  c.post_id AS post_id,
  p.creator_id as creator_id,
  date(cast(p.added_on as timestamp)) as added_on
FROM cte_content c
left JOIN oldmonkey_production.bronze_atg_post p
  ON c.post_id = p.id
WHERE p.is_alive = TRUE
),

ex_amazon as (
       select  
       ap.post_id,
       asd.brand_obj_id as brand_id,
       ap.creator_id as creator_id,
       asdp.product_id_on_brand,
      asdp.additional_info as ai1,
    --   asd.additional_info_sales as ai2,
    SUM(CASE
    WHEN CAST(asdp.total_brand_commission_earned AS DECIMAL(32,2)) > 0 and entry_type='SALE'
    THEN CAST(asdp.total_amount AS DECIMAL(32,2)) ELSE 0 END) AS sales,
       count(distinct CASE
    WHEN CAST(asdp.total_brand_commission_earned AS DECIMAL(32,2)) > 0 and entry_type='SALE'
    THEN asd.id END) as orders
       from oldmonkey_production.bronze_atg_attributedsaledataproducts asdp
       inner join oldmonkey_production.bronze_atg_attributedsaledata asd on asdp.attributed_saledata_id =asd.id
       inner join oldmonkey_production.bronze_atg_product ap on asd.product_id = ap.id
       inner join post_id p on ap.post_id = p.post_id
       where asdp.entry_type = 'SALE' 
        and ap.post_id in (select distinct post_id from post_id)
        and ap.creator_id not in (373, 581, 1916, 7435)
        and asd.short_url_click_uuid NOT LIKE '%test%' AND asd.order_id NOT IN ('000000','0')
       group by all
),

social_data as
(SELECT distinct 
  pd.post_id AS post_id,
  pd.brand_list,
  pd.boosted_flag,
  pd.total_likes AS like_count,
  pd.total_plays AS play_count,
  pd.total_comments AS comments_count,
  pd.total_shares AS total_shares,
  pd.total_views AS total_views,
  pd.total_impressions AS impressions_count,
  pd.total_reach AS reach,
  pd.total_saved AS saved,
  pd.total_watch_time_hours as total_watch_time_hours,
  pd.avg_watch_time_seconds as avg_watch_time_seconds,
  coalesce(pd.ctk_dm,0) + coalesce(pd.stk_dm,0) engage_dm,
  pd.added_on_social_media,
  pd.social_media_id
FROM business_intelligence.silver_post_data as pd
INNER JOIN post_id p
  ON pd.post_id = p.post_id
  where COALESCE(TRIM(social_media_id), '') <> ''
)
,
clicks as
(
        select p.post_id as post_id,brand_id,
        sum(total_clicks) as total_clicks
        from 
        business_intelligence.silver_product_daily spd
        inner join post_id p on spd.post_id = p.post_id
        group by all
),
unread_message as (
select vcm.id as varient_creator_id, vfm.default_value as poc, ct.id as chatthread, ct.subject, ct.admin_id, 
    ct.is_alive, ct.is_closed, ct.created_at, ct.creator_id, 
    ct.reference_id as varient_id, ct.last_message_at,
    cm.id as chatmessage_id, cm.is_read, cm.message last_unread_message, cm.is_alive, cm.sender_id, cm.thread_id, cm.created_at, cm.
    media_type, cm.updated_at, cm.sender_type, cm.template_id, cm.is_delivered, cm.message_type, cm.media_reference_id
FROM oldmonkey_production.bronze_atg_chatthread ct 
    LEFT JOIN oldmonkey_production.bronze_atg_chatmessage cm 
        ON cm.thread_id = ct.id
    left join oldmonkey_production.bronze_brandcollab_varientcreatormapping vcm
    on ct.reference_id=vcm.varient_id and ct.creator_id=vcm.creator_id
    left join oldmonkey_production.bronze_brandcollab_varientfieldmapping vfm
    on ct.reference_id=vfm.varient_id
where cm.sender_type ='CREATOR' and cm.is_read=false and vfm.field_id=19
),
creator_address as (
select distinct
    varient_creator_id,
    creator_id,
    username,
    CONCAT(
        'Name: ', delivery_name, ', ',
        'Address line: ', address_line, ', ',
        'Locality: ', locality, ', ',
        'Pincode: ', pincode, ', ',
        'City: ', city, ', ',
        'State: ', state, ', ',
        'Phone No: ', phone_number
    ) AS full_address
from (
select vcm.id varient_creator_id, vcm.creator_id, ac.username, ca.name delivery_name, ca.address_line, ca.locality, ca.pincode, ca.city, ca.state, ca.phone_number
from oldmonkey_production.bronze_brandcollab_varientcreatormapping vcm
join oldmonkey_production.bronze_atg_creator ac ON ac.id = vcm.creator_id
join (
    SELECT id, creator_id, name, address_line, locality, pincode, city, state, phone_number
    FROM oldmonkey_production.bronze_atg_creatoraddress 
    ) ca on vcm.address_id = ca.id and  ac.id = ca.creator_id
)
),
varient_type_payment as (
    select varient_id,
           default_value as payment_type
    from oldmonkey_production.bronze_brandcollab_varientfieldmapping as vfm
 where vfm.field_id=33
),
amazon_orders as (
SELECT p.post_id as post_id,
       7 as brand_id,
       t1.creator_id as creator_id,
       asin as product_id_on_brand,
       additional_info as ai1,
    --   null as ai2,
       SUM(cast(sale_amount as decimal(32,2))) AS sales,
       SUM(cast(quantity as decimal(32,2))) AS orders
FROM oldmonkey_production.bronze_atg_amazonexternalearnings as t1
right join
post_id as p on t1.creator_id=p.creator_id
WHERE
    t1.total_brand_commission_earned!=0
    and entry_type='SALE' 
    and DATEDIFF(date(cast(t1.sale_date AS timestamp)), date(p.added_on)) BETWEEN 0 AND 30 and asin in ('BYKPG')
GROUP BY all
),

sales as (
select * from ex_amazon
union 
select * from amazon_orders),

base as (
select distinct post_id, brand_id from
(
select post_id,brand_id from sales
union 
select post_id, brand_id from clicks
)),

base2 as (select p1.*, base.brand_id
from 
post_id p1
left join 
base on base.post_id=p1.post_id),

base3 as (select
        distinct
        p1.campaign_id as campaign_id,
        st.brand_list,
        p1.campaign_name,
        p1.varient_id,
        p1.varient_name,
        p1.varient_creator_id,
        ac.creator_id,
        ac.username,
        b2.post_id as post_id,
        ap.post_url,
        concat('https://www.wishlink.com/', ac.username, '/post/', CAST(ap.id AS VARCHAR(50))) AS wishlink_url,
        date(from_utc_timestamp(to_timestamp(substr(ap.added_on, 1, 16), 'yyyy-MM-dd HH:mm'), 'Asia/Kolkata')) as content_live_date,
        st.boosted_flag,
        coalesce(st.total_views,0)  as plays,
        coalesce(st.like_count,0)  as likes,
        coalesce(st.comments_count,0)  as comments,
        coalesce(st.saved,0)  as saves,
        coalesce(st.total_shares,0)  as shares,
        coalesce(st.total_views,0) AS views,
        coalesce(st.reach,0) AS reach,
        coalesce(st.total_watch_time_hours,0) as total_watch_time_hours,
        coalesce(st.avg_watch_time_seconds,0) as avg_watch_time_seconds,
        b2.brand_id,
        s.product_id_on_brand,
        s.ai1 as additional_info,
        get_json_object(s.ai1, '$.brand_on_products') AS brand_on_products,
        get_json_object(s.ai1, '$.category_on_products') AS category_on_products,
        get_json_object(s.ai1, '$.Category') AS category,
        -- s.ai2,
        -- coalesce(cl.total_click,0) as total_click,
        s.sales,
        s.orders,
        st.engage_dm
        from base2 as b2
        left join post_id p1 on p1.post_id=b2.post_id
        left join oldmonkey_production.bronze_atg_post ap on ap.id=p1.post_id
        left join sales s on b2.post_id= s.post_id and b2.brand_id=s.brand_id
        left join business_intelligence.silver_master_creator_tagging ac on ap.creator_id =ac.creator_id
        left join social_data st on ap.id = st.post_id
        where ap.is_alive =true
        order by 9 desc),
        
        clicks2 as (
        select post_id, 
               sum(total_clicks) as total_clicks,
               sum(case when brand_id=7 then total_clicks end) as amazon_clicks,
               sum(case when brand_id=227 then total_clicks end) as myntra_clicks,
               sum(case when brand_id=302 then total_clicks end) as nykaa_clicks,
               sum(case when brand_id=303 then total_clicks end) as flipkart_clicks,
               sum(case when brand_id=358 then total_clicks end) as zepto_clicks,
               sum(case when brand_id=348 then total_clicks end) as oziva_clicks
        from clicks
        group by all
        ),
        
 unique_clicks as (
        select post_id, 
               count(distinct unique_redirect) as unique_clicks
            --   sum(case when brand_id=7 then total_clicks end) as amazon_clicks,
            --   sum(case when brand_id=227 then total_clicks end) as myntra_clicks,
            --   sum(case when brand_id=302 then total_clicks end) as nykaa_clicks,
            --   sum(case when brand_id=303 then total_clicks end) as flipkart_clicks,
            --   sum(case when brand_id=358 then total_clicks end) as zepto_clicks,
            --   sum(case when brand_id=348 then total_clicks end) as oziva_clicks
        from business_intelligence.silver_product_daily_uniqueclick
        group by all
        ),
        
        
final_post as (select campaign_id,
       campaign_name,
       varient_id,
       varient_name,
       varient_creator_id,
       creator_id,
       username,
       b3.post_id,
       brand_list,
       post_url,
       wishlink_url,
       content_live_date,
       max(plays) as plays,
       max(likes) as likes,
       max(comments) as comments,
       max(saves) as saves,
       max(shares) as shares,
       max(views) as views,
       max(reach) as reach,
       max(total_watch_time_hours) as total_watch_time_hours,
       max(avg_watch_time_seconds) as avg_watch_time_seconds,
    --   product_id_on_brand,
    --   brand_on_products,
    --   category_on_products,
    --   category,
       
       --total_sales
       sum(sales) as overall_sales,
       sum(orders) as overall_orders,
       max(uc.unique_clicks) as unique_clicks,
       max(c2.total_clicks) as overall_clicks,
       sum(engage_dm) engage_dm,
       boosted_flag
    --   --amazon_sales
    --   sum(case when brand_id=7 and product_id_on_brand in ('bykpg') then sales else 0 end) as amazon_sales,
    --   sum(case when brand_id=7 and product_id_on_brand in ('bykpg') then orders else 0 end) as amazon_orders,
    --   max(c2.amazon_clicks) as amazon_clicks,
       
       
    --   --myntra sales
    --   sum(case when brand_id=227 and product_id_on_brand in ('bykpg') then sales else 0 end) as myntra_sales,
    --   sum(case when brand_id=227 and product_id_on_brand in ('bykpg') then orders else 0 end) as myntra_orders,
    --   max(c2.myntra_clicks) as myntra_clicks,
       
    --   --flipkart sales
    --   sum(case when brand_id=303 and product_id_on_brand in ('bykpg') then sales else 0 end) as flipkart_sales,
    --   sum(case when brand_id=303 and product_id_on_brand in ('bykpg') then orders else 0 end) as flipkart_orders,
    --   max(c2.flipkart_clicks) as flipkart_clicks,
       
    --   --nykaa sales
    --   sum(case when brand_id=302 and lower(brand_on_products) like '%bykpg%' then sales else 0 end) as nykaa_sales,
    --   sum(case when brand_id=302 and lower(brand_on_products) like 'bykpg' then orders else 0 end) as nykaa_orders,
    --   max(c2.nykaa_clicks) as nykaa_clicks,
       
    --   --zepto sales
    --   sum(case when brand_id=358 and lower(brand_on_products) like 'bykpg' then sales else 0 end) as zepto_sales,
    --   sum(case when brand_id=358 and lower(brand_on_products) like 'bykpg' then orders else 0 end) as zepto_orders,
    --   max(c2.zepto_clicks) as zepto_clicks,
       
    --   --oziva_sales
    --   sum(case when brand_id=348 and product_id_on_brand in ('bykpg') then sales else 0 end) as oziva_sales,
    --   sum(case when brand_id=348 and product_id_on_brand in ('bykpg') then orders else 0 end) as oziva_orders,
    --   max(c2.oziva_clicks) as oziva_clicks
       
from base3 b3
left join   
clicks2 c2 on c2.post_id=b3.post_id
left join unique_clicks uc on uc.post_id=b3.post_id
group by all),

ad_code as (
select post_id, updated_at, varient_creator_id, ads_code, adcode_final_amount
from (
select distinct post_id, ccs.updated_at, varient_creator_id, ads_code,
final_amount as adcode_final_amount,
row_number() over(partition by post_id, varient_creator_id order by DATE(ccs.updated_at) desc) rn
from oldmonkey_production.bronze_brandcollab_creatorcontentsubmission ccs
left join oldmonkey_production.bronze_brandcollab_contentadsrequest ad
on ccs.id = ad.content_id
where post_id is not null and coalesce(trim(ads_code), '') <> ''
)
where rn = 1
),

for_gmv as (select
    distinct 
    cv.campaign_id,c.is_paid,c.billing_month, c.billing_period, c.created_at campaign_launch_date, c.manager_email, c.name, c.creator_count,
    cb.varient_id,cv.brand_name, p.poc,cb.creator_id,
    mct.ig_followers as ig_followers,
    CASE 
        WHEN CAST(mct.ig_followers AS DECIMAL(38,2)) <= 5000 THEN 'Nano'
        WHEN CAST(mct.ig_followers AS DECIMAL(38,2)) BETWEEN 5001 AND 20000 THEN 'Micro'
        WHEN CAST(mct.ig_followers AS DECIMAL(38,2)) BETWEEN 20001 AND 100000 THEN 'Mid'
        WHEN CAST(mct.ig_followers AS DECIMAL(38,2)) BETWEEN 100001 AND 500000 THEN 'Macro'
        WHEN CAST(mct.ig_followers AS DECIMAL(38,2)) > 500000 THEN 'Mega'
    END AS Bucket,
    ac.username,
    cb.varient_creator_id,
    cb.campaign_status,
    -- um.last_unread_message,
    s.status,
    s.delivery_status,
    s.rejection_reason,
    pp.pitched_price,
    o.opted_in,
    o.optin_period,
    o.opted_by,
    case when lower(ack.acknowledge) like '%yes%'  then "Yes" end as acknowledgement,
    coalesce(ack.acknowledge, '') acknowledge,
    ack_period,
    ack.ack_done_by,
    fp.final_price,
    fp.final_price_date,
    vp.payment_type,
    cs.creator_selected,
    try_divide(c.amount,sum(case when lower(cs.creator_selected) like '%yes%' then fp.final_price end) over (partition by cv.campaign_id)) as ratio,
    sum(case when lower(cs.creator_selected) like '%yes%' then fp.final_price end) over (partition by cv.campaign_id) as selected_sum,
    c.amount as campaign_budget,
    fs.script_allowed,
    fs.script_submission_date,
    fs.script_submitted_by,
    fs.script_status,
    -- fs.script_approved_date,
    pd.product_delivery_status,pd.period as product_delivery_date,
    pr.product_receiving,pr.period as product_receiving_date,
    fpc.invoice_status,
    fpc.invoice_link,
    fpc.draft_approved,
    fpc.draft_added_by,
    fpc.draft_submitted,
    fpc.draft_submitted_date,
    fpc.draft_approved_date,
    case when ds2.latest_approved is not null then concat('DRAFT_APPROVED','-',ds2.latest_approved)
         when ds2.latest_rejected>ds2.latest_pending then concat('DRAFT_REJECTED','-',ds2.latest_rejected)
         else concat('DRAFT_PENDING','-',ds2.latest_pending) end as Draft_status,
    ds3.rejection_reason as draft_rejection_reason,
    fpc.content_submitted,
    fpc.content_approved,
    fpc.content_approved_by,
    fpc.content_approved_date,
    fpc.content_submitted_date,
    case when cl2.latest_approved is not null then concat('CONTENT_APPROVED','-',cl2.latest_approved)
         when cl2.latest_rejected>cl2.latest_pending then concat('CONTENT_REJECTED','-',cl2.latest_rejected)
         else concat('CONTENT_PENDING','-',cl2.latest_pending) end as Content_live_status,
    --concat(cl.status,'-',cl.period) as content_live_status,
    cld.content_submitted_by,
    r.remark,
    fpc.payouts_given,
    fpc.payout_tbd,
    fpc.payout_meta,
    fpc.ad_right_json,
    

    case when fpc.payouts_given=fpc.final_price then 'Campaign_Done' 
         when fpc.payouts_given=0 and fpc.content_approved>0 then 'Content_Approved_Full_Payout_TBD'
         when fpc.payouts_given>0 and fpc.content_approved>0 then 'Content_Approved_Second_Payout_TBD'
         when fpc.content_submitted>0 then 'Content_Submitted'
         when fpc.payouts_given<>fpc.final_price and fpc.payouts_given>0 and fpc.draft_approved>0 then 'Draft_Approved_First_Payout_Done'
         when fpc.payouts_given=0 and fpc.draft_approved>0 then 'Draft_Approved_Payout_TBD'
         when fpc.draft_submitted>0 then 'Draft_Submitted'
         when fpc.final_price<>0 and cs.creator_selected="%Yes%" then 'Creator_Selected'
         when o.opted_in like '%Yes%' then 'Creator_Opted_In'
         else 'Not Selected or Opted Out' end as execution_stage,
         
         fp1.post_id,
fp1.brand_list,
REGEXP_SUBSTR(cl2.latest_approved, 'https[^ ]+') AS submitted_content,
acc.ads_code,
acc.adcode_final_amount,
fp1.post_url,
fp1.wishlink_url,
fp1.content_live_date,
fp1.plays,
fp1.likes,
fp1.comments,
fp1.saves,
fp1.shares,
fp1.views,
fp1.reach,
fp1.total_watch_time_hours,
fp1.avg_watch_time_seconds,
fp1.overall_sales,
fp1.overall_orders,
fp1.unique_clicks,
fp1.overall_clicks,
fp1.boosted_flag,
fp1.engage_dm,
try_divide(fp1.overall_clicks,fp1.views) as CTR,
try_divide(fp.final_price,fp1.views) as CPV,
DATEDIFF(DATE(cb.creator_initiated_at), DATE(c.created_at)) launch_to_initiate_TAT,
case when opted_in ilike "%yes%" then DATEDIFF(DATE(o.optin_period),  DATE(cb.creator_initiated_at)) end optin_tat,
case
when lower(ack.acknowledge) like '%yes%' and DATEDIFF(DATE(ack.ack_period) , DATE(o.optin_period)) >= 0 then DATEDIFF(DATE(ack.ack_period) , DATE(o.optin_period)) 
when lower(ack.acknowledge) like '%yes%' and DATEDIFF(DATE(ack.ack_period) , DATE(o.optin_period)) < 0 then 0 
end as ack_tat,
case 
    when fs.script_submission_date is not null and DATEDIFF(DATE(fs.script_submission_date), DATE(ack.ack_period)) >= 0 then DATEDIFF(DATE(fs.script_submission_date), DATE(ack.ack_period))
    when fs.script_submission_date is not null and DATEDIFF(DATE(fs.script_submission_date), DATE(ack.ack_period)) < 0 then 0
end as script_tat,
case 
when draft_approved >= 1 and fs.script_submission_date is not null and DATEDIFF(DATE(fpc.draft_approved_date) , DATE(fs.script_submission_date)) >= 0 then DATEDIFF(DATE(fpc.draft_approved_date) , DATE(fs.script_submission_date))  
when draft_approved >= 1 and fs.script_submission_date is not null and DATEDIFF(DATE(fpc.draft_approved_date) , DATE(fs.script_submission_date)) < 0  then 0
when draft_approved >= 1 and fs.script_submission_date is null and DATEDIFF(DATE(fpc.draft_approved_date) , DATE(ack.ack_period)) >= 0 then DATEDIFF(DATE(fpc.draft_approved_date) , DATE(ack.ack_period))  
when draft_approved >= 1 and fs.script_submission_date is null and DATEDIFF(DATE(fpc.draft_approved_date) , DATE(ack.ack_period)) < 0 then 0
end draft_tat,
case 
when content_approved >= 1 and DATEDIFF(DATE(fpc.content_approved_date) , DATE(fpc.draft_approved_date)) >= 0 then DATEDIFF(DATE(fpc.content_approved_date) , DATE(fpc.draft_approved_date)) 
when content_approved >= 1 and DATEDIFF(DATE(fpc.content_approved_date) , DATE(fpc.draft_approved_date)) < 0 then DATEDIFF(DATE(fpc.content_approved_date) , DATE(fpc.draft_approved_date)) 
end content_approved_tat,
case 
when content_live_date is not null and DATEDIFF(DATE(content_live_date) , DATE(fpc.content_approved_date)) >= 0 then DATEDIFF(DATE(content_live_date) , DATE(fpc.content_approved_date)) 
when content_live_date is not null and DATEDIFF(DATE(content_live_date) , DATE(fpc.content_approved_date)) < 0 then 0
end content_live_tat,
case
when cs.creator_selected ilike '%yes%' and DATEDIFF(DATE(cs.period), DATE(c.created_at)) >= 0 then DATEDIFF(DATE(cs.period), DATE(c.created_at))
when cs.creator_selected ilike '%yes%' and DATEDIFF(DATE(cs.period), DATE(c.created_at)) < 0 then 0
end as launch_to_sel_tat,
case 
when draft_submitted >= 1 and DATEDIFF(DATE(fpc.draft_submitted_date), DATE(cs.period)) >= 0 then DATEDIFF( DATE(fpc.draft_submitted_date), DATE(cs.period))
when draft_submitted >= 1 and DATEDIFF(DATE(fpc.draft_submitted_date), DATE(cs.period)) < 0 then 0
end as selection_to_draft_sub_tat,
case 
when draft_approved >= 1 and DATEDIFF(DATE(fpc.draft_approved_date), DATE(cs.period)) >= 0 then DATEDIFF( DATE(fpc.draft_approved_date), DATE(cs.period))
when draft_approved >= 1 and DATEDIFF(DATE(fpc.draft_approved_date), DATE(cs.period)) < 0 then 0
end as selection_to_draft_app_tat,
case 
when content_live_date is not null and DATEDIFF(DATE(content_live_date), DATE(fpc.draft_submitted_date)) >= 0 then DATEDIFF( DATE(fpc.draft_approved_date), DATE(cs.period))
when content_live_date is not null and DATEDIFF(DATE(content_live_date), DATE(fpc.draft_submitted_date)) < 0 then 0
end as draft_sub_to_content_live_tat,
case 
when content_live_date is not null and DATEDIFF(DATE(content_live_date), DATE(fpc.draft_approved_date)) >= 0 then DATEDIFF( DATE(fpc.draft_approved_date), DATE(cs.period))
when content_live_date is not null and DATEDIFF(DATE(content_live_date), DATE(fpc.draft_approved_date)) < 0 then 0
end as draft_app_to_content_live_tat,
ca.full_address,
tagging_results,
transcription_language,
transcription_duration,
transcription_text,
tagged_transcript_language,
valid_transcript,
relevant_transcript,
transcript_summary,
frame_overall_score,
frame_description,
text_overlay,
background_score,
background_explanation,
composition_score,
composition_explanation,
lighting_score,
lighting_explanation,
aesthetic_appeal_score,
aesthetic_appeal_explanation,
styling_score,
styling_explanation,
content_polish_score,
content_polish_explanation,
contextual_relevance_score,
contextual_relevance_explanation,
marketing_fit_score,
marketing_fit_explanation,
product_visibility_score,
product_visibility_explanation,
storytelling_score,
storytelling_explanation,
summary_overall_score,
content_category,
content_subcategory,
content_audience,
strengths,
improvements,
frame_keywords,
frame_topics,
product_name,
product_description,
product_brand
from 
    creator_base cb
left join 
    status s
on cb.varient_creator_id=s.varient_creator_id
left join pitched_price pp  on cb.varient_creator_id=pp.varient_creator_id
left join opt o  on cb.varient_creator_id=o.varient_creator_id
left join final_price fp  on cb.varient_creator_id=fp.varient_creator_id
left join final_script fs on cb.varient_creator_id = fs.varient_creator_id
left join creator_selection cs  on cb.varient_creator_id=cs.varient_creator_id
left join product_delivery pd  on cb.varient_creator_id=pd.varient_creator_id
left join product_receiving pr  on cb.varient_creator_id=pr.varient_creator_id
left join draft_status2 ds2  on cb.varient_creator_id=ds2.varient_creator_id
left join content_live2 cl2  on cb.varient_creator_id=cl2.varient_creator_id
left join content_live_done_by cld on cb.varient_creator_id=cld.varient_creator_id
left join oldmonkey_production.bronze_atg_creator ac on ac.id=cb.creator_id
left join remark r on r.varient_creator_id=cb.varient_creator_id
left join draft_status3 ds3 on cb.varient_creator_id=ds3.varient_creator_id
left join final_payouts_cte fpc on fpc.varient_creator_id=cb.varient_creator_id
-- left join unread_message um on um.varient_creator_id = cb.varient_creator_id
left join oldmonkey_production.bronze_brandcollab_campaignvarient cv on cv.id=cb.varient_id
left join oldmonkey_production.bronze_brandcollab_campaign c on c.id=cv.campaign_id
left join poc p on p.varient_id=cb.varient_id
left join final_post as fp1 on fp1.varient_creator_id=cb.varient_creator_id
left join acknowledged as ack on ack.varient_creator_id=cb.varient_creator_id
left join business_intelligence.silver_master_creator_tagging mct on mct.creator_id=cb.creator_id
left join creator_address ca on ca.varient_creator_id = cb.varient_creator_id
left join varient_type_payment vp on vp.varient_id = cb.varient_id
left join wishlink_datazip.brandcollab_post_base pb on fp1.post_id = pb.post_id
left join ad_code acc on fp1.varient_creator_id = acc.varient_creator_id  and fp1.post_id = acc.post_id
order by 3)

select g.*,sum(spd2.commissionable_gmv) as commissionable_gmv, sum(spd2.non_commissionable_gmv) as non_commissionable_gmv, 
sum(spd2.non_commissionable_orders) as non_commissionable_orders
from for_gmv as g 
left join business_intelligence.silver_product_daily spd2
on g.creator_id=spd2.creator_id and g.post_id=spd2.post_id
-- where g.is_paid=true  
group by all
)